# 17. 数组基础（ndarray）

<!-- module-learning-arc:start -->
> **NumPy 模块主线｜第 2 / 6 步：建立 ndarray 数据模型**
>
> **持续应用背景：** 为区域仓库建立补货预警矩阵：把门店、商品、库存和需求组织成数组，逐步完成定位、广播计算、排序和抽样复核。
>
> **承接上一阶段：** NumPy 模块入门  →  **本章任务：** 数组基础（ndarray）  →  **下一步：** 索引、切片与筛选
>
> **大作业连接：** 本章练习将成为《连锁门店补货预警矩阵》的一部分，最终需要把数组建模、风险筛选、广播计算和抽样复核组合成一份可执行的补货清单。
<!-- module-learning-arc:end -->


## 本章场景

上一阶段你用 Python 列表管账，一次只能对一个数做运算（`sales[0] * 1.1`、`sales[1] * 1.1`……）。现在数据量一上来——几千条商品单价 × 数量——列表就扛不住了：循环慢、写法啰嗦、还容易出错。

**NumPy 数组**把这些数放进一个"整批计算"的容器：创建时一次搞清楚它的**形状**（shape）、**维数**（ndim）、**类型**（dtype），后面就能对整批数据一起加税、一起求最值。本章先学会"把数据装进数组、看清它长什么样、控制它的类型"，这是后面索引、广播、统计的地基。


## 本章目标

学完本章，你将能够：

- **理解**：理解 ndarray 的 shape / ndim / dtype 三个属性，以及"同一类型、整批计算"的意义。
- **操作**：能用 np.array / arange / linspace / zeros / ones 创建数组，用 reshape 重排、astype 转换类型。
- **迁移**：能把一批业务数字装进数组，并准确确认它的形状与类型，为后续按条件筛选、按轴汇总打好基础。


## 17.1 核心概念

**背景引入**：数组是整批数据的“统一容器”，先记住三个属性就够用——**shape**（形状）回答“排成几行几列”，**ndim**（维数）回答“有几个维度”，**dtype**（类型）回答“里面放的是哪种数”。一维像一列数，二维像一张表，读到逗号就多一个维度——记住这条口诀，数组的“形状”就不会再绕晕你。

- NumPy数组通常存储同一类型的数据。
- shape描述各维长度，ndim描述维数。
- 固定数值类型能提高运算效率并减少隐式转换。


## 17.2 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| np.array() | `np.array()` | array把Python序列转换为NumPy数组，并提供shape和dtype等属性。 | 把shape和size混为一谈 |
| zeros() 和 ones() | `np.zeros()`、`np.ones()` | 用固定值创建指定形状的数组，适合初始化矩阵。 | 整数类型保存小数导致精度丢失 |
| arange() | `np.arange()` | arange生成等步长序列，右端点通常不包含。 | 创建不规则嵌套列表数组 |
| linspace() | `np.linspace()` | linspace按点数控制结果，适合绘图和连续模拟。 | 把shape和size混为一谈 |
| reshape() | `np.arange()`、`values.reshape()` | reshape要求元素总数不变，只改变数组的组织方式。 | 整数类型保存小数导致精度丢失 |
| astype() | `np.array()`、`values.astype()` | astype返回转换后的新数组，原数组不会被原地修改。 | 创建不规则嵌套列表数组 |


## 17.3 示例 1：创建数组并检查属性

**背景引入**：处理一批销售数字时，Python 列表能用，但要对整批数据做"每个都加税""求最大值"这类运算要写循环。NumPy 数组把整批数据变成一块可整体计算的对象，创建时的第一件事就是认识它的三个属性——shape（形状）、ndim（维数）、dtype（类型）。

**讲解**：数组属性帮助确认后续运算需要的结构。

- shape 是各维长度的元组：`np.array([1,2,3])` 的 shape 是 `(3,)`，`np.array([[1,2],[3,4]])` 是 `(2,2)`；
- ndim 等于 shape 的长度：一维 ndim=1，二维 ndim=2（`matrix.shape[1]` 才是列数）；
- dtype 描述元素类型：`np.float64` 能存小数，`int` 不能；
- **口诀**：一维像一列数，二维像一张表，读到逗号就多一个维度。


<!-- math-foundation:chapter-17 -->
### 数学推导｜数组的形状就是坐标系统

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先确定轴。** $m\times n$ 数组包含 $m$ 行与 $n$ 列，总元素数为

$$
N=mn
$$

**第 2 步｜用两个下标定位。** 元素 $A_{ij}$ 的第一个下标沿行轴变化，第二个下标沿列轴变化。

**第 3 步｜连接到内存中的线性位置。** 对按行连续存放的数组，数学下标从 0 计时，线性位置可写成

$$
k=in+j
$$

所以“形状”不仅描述大小，也决定一维数据如何被解释成坐标。

**把上面的关系收束为本章计算式：**

$$
A\in\mathbb{R}^{m\times n},\qquad A_{ij}=\text{第 }i\text{ 行、第 }j\text{ 列的值}
$$

**符号解释：** $m$ 是行数，$n$ 是列数；二维数组的 `shape` 为 $(m,n)$。

**代码对应：** `A.shape == (m, n)`，`A[i, j]` 访问数学记号中的 $A_{ij}$（Python 下标从 0 开始）。

**使用边界：** 数学下标常从 1 开始，NumPy 下标从 0 开始，解释位置时要明确约定。


In [ ]:
import numpy as np

sales = np.array([120, 150, 180, 210], dtype=np.float64)
matrix = np.array([[1, 2, 3], [4, 5, 6]])
print(sales)
print("shape:", matrix.shape)
print("ndim:", matrix.ndim)
print("dtype:", sales.dtype)


## 17.4 示例 2：规则序列

**背景引入**：很多数组不是手动一个个写数字，而是"从 1 数到 7""从 5% 到 20% 均匀取 4 个点"这类**有规律**的序列。按规则生成，既省去手工输入又不容易出错。

**讲解**：arange 控制步长，linspace 控制点数。

- `np.arange(1, 8)`：从 1 到 7，步长为 1（右端点 8 不包含）；
- `np.linspace(0.05, 0.20, 4)`：在 0.05 到 0.20 之间均匀取 4 个点（含两端）；
- `np.zeros((2,3))`：2 行 3 列全 0；`np.ones(5, dtype=int)`：5 个全 1 的整数；
- **口诀**：要"步长"用 arange，要"几个点"用 linspace，要"形状占位"用 zeros/ones。


In [ ]:
days = np.arange(1, 8)
rates = np.linspace(0.05, 0.20, 4)
zeros = np.zeros((2, 3))
ones = np.ones(5, dtype=int)
print(days)
print(rates)
print(zeros)
print(ones)


## 17.5 示例 3：类型转换

**背景引入**：数据导入时价格常是文本（"128.5"）而不是数字。文本不能直接做加减乘除，必须先转成数值；而整数类型又存不了小数。类型转换是数据进入运算前的必经一步。

**讲解**：astype 返回新数组，转换前应检查是否会损失精度。

- `text_array.astype(float)`：把文本数组转成浮点数组，`"128.5"` 变成 `128.5`；
- `float_array.astype(int)`：把浮点转成整数，`128.5` 变成 `128`（**截断小数，可能丢精度**）；
- astype 返回**新数组**，不修改原数组；
- **口诀**：转类型会丢精度，先算后转；只要不是存小数，优先用整数省内存。


In [ ]:
price_text = np.array(["128.5", "299.0", "59.9"])
prices = price_text.astype(float)
rounded = prices.astype(int)
print(prices, prices.dtype)
print(rounded, rounded.dtype)


## 17.6 核心操作独立示例

下面每个代码单元格只演示一个核心方法、函数或语法操作。请先阅读方法名称和任务说明，再单独运行当前单元格；示例尽量自带最小输入，不要求依赖前一个单元格留下的变量。


In [ ]:
# np.array()
# array把Python序列转换为NumPy数组，并提供shape和dtype等属性。
import numpy as np

values = np.array([10, 20, 30])
print(values)
print("shape:", values.shape)
print("dtype:", values.dtype)


In [ ]:
# zeros() 和 ones()
# 用固定值创建指定形状的数组，适合初始化矩阵。
import numpy as np

print(np.zeros((2, 3)))
print(np.ones((2, 3)))


In [ ]:
# arange()
# arange生成等步长序列，右端点通常不包含。
import numpy as np

print(np.arange(0, 10, 2))


In [ ]:
# linspace()
# linspace按点数控制结果，适合绘图和连续模拟。
import numpy as np

print(np.linspace(0, 1, 5))


In [ ]:
# reshape()
# reshape要求元素总数不变，只改变数组的组织方式。
import numpy as np

values = np.arange(12)
matrix = values.reshape(3, 4)
print(matrix)
print(matrix.shape)


In [ ]:
# astype()
# astype返回转换后的新数组，原数组不会被原地修改。
import numpy as np

values = np.array([1.2, 2.8, 3.4])
integers = values.astype(int)
print(integers)
print(integers.dtype)


**练一练 14.6**：创建数组 arr = np.arange(1, 13).reshape(3, 4)，完成三件事：打印 arr 的形状与数据类型；用布尔筛选取出所有大于 6 的元素；把第 2 行第 3 列的元素改成 0 后打印。


In [ ]:
# 请在下方填写代码
import numpy as np

arr = np.arange(1, 13).reshape(3, 4)
print(arr.shape, arr.dtype)
print(arr[arr > 6])
arr[1, 2] = 0
print(arr)


**输出解读**：`arr.shape` 是 `(3, 4)`、`arr.dtype` 是 `int64`；`arr[arr > 6]` 返回满足条件的一维数组 `[ 7  8  9 10 11 12]`；执行 `arr[1, 2] = 0` 后，第 2 行第 3 列变成 0（原数组被**原地修改**）。注意：布尔筛选得到的是**新数组**，而索引赋值改写的是**原数组**——一个"取出来用"，一个"改回去"，二者不要混。


## 17.7 独立迁移练习

先预测 shape，再修改一个数组或筛选条件，解释结果变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# TODO: 在此粘贴或改写最接近的示例。
# 记录：我改了什么？预期会发生什么？实际观察到什么？
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print({"修改": change_note, "预期": expected_change, "观察": observed_change})


## 17.8 本章实训：axis与布尔筛选

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np

matrix = np.arange(1, 13).reshape(3, 4)
print("原数组：\n", matrix)
print("每行合计：", matrix.sum(axis=1))
print("每列合计：", matrix.sum(axis=0))


### 17.8.1 第一个结果怎么读

`axis=1` 保留行，沿列方向计算；`axis=0` 保留列，沿行方向计算。先看 shape，再解释结果长度。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
even = matrix[matrix % 2 == 0]
print("偶数：", even)
print("偶数数量：", even.size)
print("偶数平均值：", even.mean())


### 17.8.2 第二个结果怎么读

第二个实验不改原数组，而是用布尔条件筛选新数组。请思考：如果条件改成 `matrix > 8`，输出会怎样变化？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 17.9 错误恢复：数组形状不匹配怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import numpy as np

matrix = np.arange(6).reshape(2, 3)
try:
    result = matrix + np.array([10, 20])
except ValueError as error:
    print("形状问题：", type(error).__name__)
    result = matrix + np.array([10, 20, 30])
print("修复后的结果：")
print(result)


### 17.9.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

先看两个数组的 shape，再判断能否广播。修复不是随意 reshape，而是让数据结构和业务含义一致。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 17.10 易错点提醒

- 把shape和size混为一谈
- 整数类型保存小数导致精度丢失
- 创建不规则嵌套列表数组


## 17.11 练习与作业

1. 创建3×4数组
2. 输出维度、形状和元素数
3. 转换为浮点类型

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 17.12 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“创建3×4数组”。
2. **独立完成**：不复制示例代码，完成“输出维度、形状和元素数”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“转换为浮点类型”，用一两句话说明你修改了什么。

### 17.12.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 17.12.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import numpy as np

# TODO: 创建3×4数组，元素为1到12
# TODO: 转换为浮点类型
# TODO：请在下方完成 —— 14.12 练习与作业 1. 创建3×4数组 2. 输出维度、形状和元素数 3. 转换为浮点类型 提交前检查：代码可从上


In [ ]:
import numpy as np

arr = np.arange(1, 13).reshape(3, 4)
float_arr = arr.astype(float)
print(arr)
print("ndim:", arr.ndim, "shape:", arr.shape, "size:", arr.size)
print(float_arr.dtype)


## 17.13 小结

认识ndarray、形状和数据类型，使用多种方式创建数值数组。

**迁移思考**：

1. 如果需要创建一个 2×3×4 的三维数组，shape 应该是什么？ndim 是多少？
2. 为什么 NumPy 数组要求同一类型？这种限制带来了什么好处？


### 17.13.1 你已经掌握

- 创建一维和多维数组
- 读取shape、ndim和dtype
- 使用arange与linspace
- 控制数组数据类型


### 17.13.2 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 17.13.3 需要注意

- 把shape和size混为一谈
- 整数类型保存小数导致精度丢失
- 创建不规则嵌套列表数组


### 17.13.4 完成检查

- [ ] 能够创建一维和多维数组
- [ ] 能够读取shape、ndim和dtype
- [ ] 能够使用arange与linspace
- [ ] 能够控制数组数据类型


### 17.13.5 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
